In [1]:
import pandas as pd
import os
import numpy as np
import geopandas as gpd

#### Definição de caminho e CRS

In [18]:
repo_path = r'C:\Users\gabriel.coimbra\Desktop\Concórdia\Materiais criados\Análise Ligações'
ligacoes_caminho = os.path.join(repo_path, 'ligacoes_class.xlsx')
bacias_caminho = r'C:\Users\gabriel.coimbra\Desktop\Concórdia\Materiais criados\BaciasH_Todas_V0.gpkg'
setores_caminho = r'C:\Users\gabriel.coimbra\Desktop\Concórdia\Cálculo de população\SC_setores_CD2022.gpkg'
crs = "EPSG:31982"

#### Importação de DF de ligações + filtro de ligações ativas

In [4]:
#Ligações classificadas por bacia
df = pd.read_excel(ligacoes_caminho)

# Filtrar apenas ligações ativas
df_ativa = df[df['SITUACAO_L'] == 'ATIVA'].copy()

#transformando df em gdf
gdf_ativa = gpd.GeoDataFrame(
    df_ativa,
    geometry=gpd.points_from_xy(df_ativa["LONG"], df_ativa["LAT"]),
    crs="EPSG:4326"
)

#convertendo pra sistema de coordenadas padrão
gdf_ativa = gdf_ativa.to_crs(crs)

#### Importação das bacias

In [19]:
#lendo arquivo de bacias
bacias = gpd.read_file(bacias_caminho)

#convertendo pra sistema de coordenadas padrão
bacias = bacias.to_crs(crs) 

c:\Users\gabriel.coimbra\Documents\GitHub\EngSanitariaAmbiental\venv\Lib\site-packages\pyogrio\raw.py:198: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D MultiPolygon' is converted to 'MultiPolygon Z'
  return ogr_read(


In [20]:
bacias

,id,Nome,OBJECTID,Shape_Leng,Sistema,Sub_bacia,Shape_Length,Shape_Area,layer,path,Setor,geometry
0,NaN,Bacia BRF,NaN,NaN,None,None,NaN,NaN,Bacias_SES_Central,C:/Users/gabriel.coimbra/Desktop/Concórdia/SIG...,BRF,"MULTIPOLYGON Z (((396265.621 6988886.56 0, 396..."
1,NaN,Bacia 4B-1,NaN,NaN,None,None,NaN,NaN,Bacias_SES_Central,C:/Users/gabriel.coimbra/Desktop/Concórdia/SIG...,Central,"MULTIPOLYGON Z (((398191.997 6985629.939 0, 39..."
2,NaN,Bacia 4B-2,NaN,NaN,None,None,NaN,NaN,Bacias_SES_Central,C:/Users/gabriel.coimbra/Desktop/Concórdia/SIG...,Central,"MULTIPOLYGON Z (((398802.384 6986321.6 0, 3987..."
3,NaN,Bacia 4B,NaN,NaN,None,None,NaN,NaN,Bacias_SES_Central,C:/Users/gabriel.coimbra/Desktop/Concórdia/SIG...,Central,"MULTIPOLYGON Z (((398600.546 6986094.26 0, 398..."
4,NaN,Bacia 5,NaN,NaN,None,None,NaN,NaN,Bacias_SES_Central,C:/Users/gabriel.coimbra/Desktop/Concórdia/SIG...,Central,"MULTIPOLYGON Z (((399027.339 6986149.099 0, 39..."
5,NaN,Bacia 5B-1,NaN,NaN,None,None,NaN,NaN,Bacias_SES_Central,C:/Users/gabriel.coimbra/Desktop/Concórdia/SIG...,Central,"MULTIPOLYGON Z (((398555.069 6988755.457 0, 39..."
6,NaN,Bacia N1,41.0,1705.168765,Fragosos,SBF-11,1705.168765,126583.150109,SES Natureza,C:/Users/gabriel.coimbra/Desktop/Concórdia/Mat...,Natureza,"MULTIPOLYGON Z (((394954 6989547 0, 394963 698..."
7,NaN,Bacia N2,42.0,1991.961833,Fragosos,SBF-10,1991.961833,149380.577518,SES Natureza,C:/Users/gabriel.coimbra/Desktop/Concórdia/Mat...,Natureza,"MULTIPOLYGON Z (((394595 6989782 0, 394600 698..."


#### Importação dos setores do IBGE

In [9]:
#lendo arquivo de setores censitários
setores = gpd.read_file(setores_caminho)

#Filtrando os que possuem algum domicílio (para não dar erro)
setores = setores[setores['v0003']>0]

#convertendo pra sistema de coordenadas padrão
setores = setores.to_crs(crs) 

#### Classificação das ligações por bacia



In [22]:
ligacoes_bacias = gpd.sjoin(
    gdf_ativa,
    bacias[['Nome','Setor','geometry']],
    how = 'left',
    predicate='within'
)

Setor
Central     5810
BRF          784
Natureza     262
Name: count, dtype: int64

In [33]:
# Ligações fora das bacias são consideradas "outros"
ligacoes_bacias['Nome'] = np.where(
    ligacoes_bacias['Nome'].isna() | (ligacoes_bacias['Nome'].astype(str).str.strip() == ''),
    'Outro',
    ligacoes_bacias['Nome']
)

#O campo 'descriptio' é relacionado ao distrito. Se não tiver em nenhum, é classificado como "Outro"
ligacoes_bacias['Setor'] = np.where(
    ligacoes_bacias['Setor'].isna() | (ligacoes_bacias['Setor'].astype(str).str.strip() == ''),
    'Outros',
    ligacoes_bacias['Setor']
)

ligacoes_bacias['Setor'].value_counts()

Setor
Outros      14420
Central      5810
BRF           784
Natureza      262
Name: count, dtype: int64

# CONTINUAR ANALISE DE LIGAÇÕES
erro index

In [34]:
ligacoes_bacias.drop(columns= 'index_right')
ligacoes_bacias.columns

Index(['LIGACAO', 'GRUPO', 'ROTA', 'QUADRA', 'LOTE', 'UNIDADE', 'BAIRRO',
       'ENDERECO', 'CEP_PRINCI', 'SITUACAO_L', 'TIPO_FATUR', 'COBRANCA_F',
       'DATA_DA_LI', 'DATA_DA__1', 'HIDROMETRO', 'VAZAO', 'CAPACIDADE',
       'MARCA', 'ANO_DE_FAB', 'DATA_DE_IN', 'RESIDENCIA', 'COMERCIAL',
       'PUBLICO', 'INDUSTRIAL', 'TOTAL_ECON', 'CATEGORIA', 'DESTINATAR',
       'LOUGRADOUR', 'CEP_ENTREG', 'BAIRRO_ENT', 'CIDADE_ENT', 'UF_ENTREGA',
       'TIPO_DE_EN', 'LAT', 'LONG', 'OBS COORD', 'SETOR_ID', 'SETOR_BAIR',
       'fid', 'Name', 'descriptio', 'timestamp', 'begin', 'end', 'altitudeMo',
       'tessellate', 'extrude', 'visibility', 'drawOrder', 'icon', 'geometry',
       'index_right', 'Nome', 'Setor'],
      dtype='object')

In [36]:
#classificando cada ligação com base no setor censitário no qual está inserido
#v0005 é a densidade com base na população e domicílios ocupados
ligacoes_bacias_setores = gpd.sjoin(
    ligacoes_bacias,
    setores[['CD_SETOR', 'v0005','geometry']],
    how='left',
    predicate='within').drop(columns=["index_right"])

ligacoes_bacias_setores['populacao'] = ligacoes_bacias_setores['RESIDENCIA']*ligacoes_bacias_setores['v0005']

#transformando em df pq agr n preciso mais da georeferrência
ligacoes_bacias_setores = ligacoes_bacias_setores.drop(columns="geometry")

ValueError: 'index_right' cannot be a column name in the frames being joined

In [ ]:
# Colunas para agregar
cols = ['RESIDENCIA', 'COMERCIAL', 'PUBLICO', 'INDUSTRIAL', 'TOTAL_ECON']

# Garantir que as colunas estejam numéricas
ligacoes_bacias_setores[cols] = ligacoes_bacias_setores[cols].apply(pd.to_numeric, errors='coerce').fillna(0)

# 1) Versão somando as colunas
df_soma = (
    ligacoes_bacias_setores
    .groupby(['TIPO_FATUR', 'Name','descriptio'], as_index=False)[cols + ['populacao']]
    .sum()
)

df_soma = df_soma.rename(columns={
    'TIPO_FATUR': 'Tipo de faturamento',
    'Name': 'Bacia',
    'populacao':'População',
    'RESIDENCIA': 'Eco_Residencial',
    'COMERCIAL': 'Eco_Comercial',
    'PUBLICO': 'Eco_Público',
    'INDUSTRIAL': 'Eco_Industrial',
    'TOTAL_ECON': 'Eco_Total'
})

# 2) Versão contando valores diferentes de zero
df_contagem_sem_zero = (
    df_ativa
    .groupby(['TIPO_FATUR', 'Name','descriptio'])[cols]
    .agg(lambda x: (x != 0).sum())
    .reset_index()
)

df_contagem_sem_zero = df_contagem_sem_zero.rename(columns={
    'TIPO_FATUR': 'Tipo de faturamento',
    'Name': 'Bacia',
    'RESIDENCIA': 'Lig_Residencial',
    'COMERCIAL': 'Lig_Comercial',
    'PUBLICO': 'Lig_Público',
    'INDUSTRIAL': 'Lig_Industrial',
    'TOTAL_ECON': 'Lig_Total'
})

df_merge = pd.merge(
    df_soma,
    df_contagem_sem_zero,
    on=['Tipo de faturamento', 'Bacia','descriptio'],
    how='outer'
)

df_merge= df_merge[['descriptio','Bacia','População','Tipo de faturamento','Eco_Residencial','Lig_Residencial',
                   'Eco_Comercial','Lig_Comercial','Eco_Público','Lig_Público',
                   'Eco_Industrial','Lig_Industrial','Eco_Total','Lig_Total']]


df_merge = df_merge.sort_values(
    by=['Bacia', 'Tipo de faturamento']
).reset_index(drop=True)


In [ ]:
# Exportar
df_merge.to_excel(os.path.join(repo_path, 'resultado.xlsx'), index=False)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import numpy as np
import os
plt.rcParams['font.family'] = 'Arial'

def gerar_figura_barras_empilhadas_categoria_sem_outro(
    df_merge,
    coluna_classificacao='Bacia',
    repo_path=None,
    salvar=True,
    nome_arquivo='figura_barras_empilhadas_categoria_sem_outro.png',
    remover_outro=True
):
    df_plot = df_merge.copy()

    # Remover "Outro", se existir na coluna escolhida
    if remover_outro:
        df_plot = df_plot[df_plot[coluna_classificacao] != 'Outro'].copy()

    # Soma todos os tipos de faturamento dentro da coluna escolhida
    df_bacia = (
        df_plot
        .groupby(coluna_classificacao, as_index=False)[[
            'Eco_Residencial', 'Eco_Comercial', 'Eco_Público', 'Eco_Industrial',
            'Lig_Residencial', 'Lig_Comercial', 'Lig_Público', 'Lig_Industrial'
        ]]
        .sum()
    )

    df_bacia = df_bacia.sort_values(coluna_classificacao).reset_index(drop=True)

    categorias = ['Residencial', 'Comercial', 'Público', 'Industrial']

    cols_eco = {
        'Residencial': 'Eco_Residencial',
        'Comercial': 'Eco_Comercial',
        'Público': 'Eco_Público',
        'Industrial': 'Eco_Industrial'
    }

    cols_lig = {
        'Residencial': 'Lig_Residencial',
        'Comercial': 'Lig_Comercial',
        'Público': 'Lig_Público',
        'Industrial': 'Lig_Industrial'
    }

    cores = {
        'Residencial': '#4C78A8',
        'Comercial': '#F58518',
        'Público': '#54A24B',
        'Industrial': '#B279A2'
    }

    def formatar_milhar(x, pos):
        return f'{int(x):,}'.replace(',', '.')

    bacias = df_bacia[coluna_classificacao].tolist()

    x = np.arange(len(bacias))
    largura = 0.36

    # Barras de economias e ligações coladas
    x_eco = x - largura / 2
    x_lig = x + largura / 2

    base_eco = np.zeros(len(bacias))
    base_lig = np.zeros(len(bacias))

    total_eco = df_bacia[[cols_eco[c] for c in categorias]].sum(axis=1).values
    total_lig = df_bacia[[cols_lig[c] for c in categorias]].sum(axis=1).values

    max_total = max(total_eco.max(), total_lig.max())

    fig, ax = plt.subplots(figsize=(8, 6))

    for categoria in categorias:
        valores_eco = df_bacia[cols_eco[categoria]].values
        valores_lig = df_bacia[cols_lig[categoria]].values

        ax.bar(
            x_eco,
            valores_eco,
            width=largura,
            bottom=base_eco,
            label=categoria,
            color=cores[categoria],
        )

        ax.bar(
            x_lig,
            valores_lig,
            width=largura,
            bottom=base_lig,
            color=cores[categoria],
            edgecolor='white',
            linewidth=0.6
        )

        # Percentuais dentro das barras de economias
        for i, valor in enumerate(valores_eco):
            if total_eco[i] > 0:
                perc = valor / total_eco[i] * 100

                if perc > 5:
                    ax.text(
                        x_eco[i],
                        base_eco[i] + valor / 2,
                        f'{perc:.1f}%',
                        ha='center',
                        va='center',
                        fontsize=10,
                        color='white'
                    )

        # Percentuais dentro das barras de ligações
        for i, valor in enumerate(valores_lig):
            if total_lig[i] > 0:
                perc = valor / total_lig[i] * 100

                if perc > 5:
                    ax.text(
                        x_lig[i],
                        base_lig[i] + valor / 2,
                        f'{perc:.1f}%',
                        ha='center',
                        va='center',
                        fontsize=10,
                        color='white',
                    )

        base_eco += valores_eco
        base_lig += valores_lig

    # Totais no topo das barras
    for x_pos, total in zip(x_eco, total_eco):
        if total > 0:
            ax.text(
                x_pos,
                total + max_total * 0.018,
                f'{int(total):,}'.replace(',', '.'),
                ha='center',
                va='bottom',
                fontsize=11,
                fontweight='bold'
            )

    for x_pos, total in zip(x_lig, total_lig):
        if total > 0:
            ax.text(
                x_pos,
                total + max_total * 0.018,
                f'{int(total):,}'.replace(',', '.'),
                ha='center',
                va='bottom',
                fontsize=11,
                fontweight='bold'
            )

    # Tirar eixo X padrão
    ax.set_xticks([])
    ax.set_xticklabels([])
    
    # Colocar Eco/Lig em cima do nome da bacia
    for i, bacia in enumerate(bacias):
        y_eco_lig = -max_total * 0.025
        y_bacia = -max_total * 0.095
    
        ax.text(
            x_eco[i],
            y_eco_lig,
            'Eco',
            ha='center',
            va='top',
            fontsize=11
        )
    
        ax.text(
            x_lig[i],
            y_eco_lig,
            'Lig',
            ha='center',
            va='top',
            fontsize=11
        )
    
        ax.text(
            x[i],
            y_bacia,
            bacia,
            ha='center',
            va='top',
            fontsize=12,
            fontweight='bold'
        )
    
    # Tirar eixo Y
    ax.set_ylabel('')
    ax.set_yticks([])
    ax.set_yticklabels([])
    ax.tick_params(axis='y', length=0)

    # Tirar eixo X
    ax.tick_params(axis='x', length=0)

    # Tirar grid
    ax.grid(False)

    # Tirar box da figura
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.spines['left'].set_visible(False)
    # Sem título
    ax.set_title('')

    # Legenda no canto, em uma linha
    ax.legend(
        loc='upper left',
        bbox_to_anchor=(0.8, 0.95),
        ncol=1,
        frameon=False,
        fontsize=12,
        title_fontsize=12,
        handlelength=1.6,
        columnspacing=1.4
    )

    ax.set_ylim(0, max_total * 1.18)
    #ax.margins(x=0.04)


    plt.tight_layout()

    if salvar and repo_path is not None:
        caminho_saida = os.path.join(repo_path, nome_arquivo)
        fig.savefig(caminho_saida, dpi=300, bbox_inches='tight')
        print(f'Figura salva em: {caminho_saida}')

    plt.show()
    
gerar_figura_barras_empilhadas_categoria_sem_outro(
    df_merge,
    coluna_classificacao='Bacia',
    repo_path=repo_path,
    salvar=True,
    nome_arquivo='figura_por_bacia.png'
)

df_regioes = df_merge.groupby(['descriptio','Tipo de faturamento'],as_index=False).sum()
df_regioes.to_excel(os.path.join(repo_path, 'resultado_df_regioes.xlsx'), index=False)

#gerar por região do SES
gerar_figura_barras_empilhadas_categoria_sem_outro(
    df_regioes,
    coluna_classificacao='descriptio',
    repo_path=repo_path,
    salvar=True,
    nome_arquivo='regiao.png'
)

df_centro = df_merge[df_merge['descriptio']=='Centro']